In [3]:
from embedder import Embedder

embed = Embedder()

v = embed.encode("Hello world")

print(v.shape)

(384,)


In [4]:
# Q1 – Embedding a Query.
from embedder import Embedder

embed = Embedder()

query = "How does approximate nearest neighbor search work?"

query_vector = embed.encode(query)

print(f"Embedding shape: {query_vector.shape}")
print(f"First value: {query_vector[0]}")

Embedding shape: (384,)
First value: -0.02058203437252893


In [5]:
# Q2. Cosine similarity
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

len(documents)

72

In [6]:
for doc in documents:
    print(doc["filename"])

01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/02-environment.md
01-agentic-rag/lessons/03-rag.md
01-agentic-rag/lessons/04-dataset.md
01-agentic-rag/lessons/05-search.md
01-agentic-rag/lessons/06-building-prompt.md
01-agentic-rag/lessons/07-llm.md
01-agentic-rag/lessons/08-rag-helper.md
01-agentic-rag/lessons/09-data-ingestion.md
01-agentic-rag/lessons/10-rag-next-steps.md
01-agentic-rag/lessons/11-agents-intro.md
01-agentic-rag/lessons/12-rag-revision.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/16-other-frameworks.md
02-vector-search/lessons/01-intro.md
02-vector-search/lessons/02-embeddings.md
02-vector-search/lessons/03-embeddings-dataset.md
02-vector-search/lessons/04-vector-search.md
02-vector-search/lessons/05-minsearch-vector.md
02-vector-search/lessons/06-rag-vector.md
02-vector-search/lessons/07-sqlitesearch-vector.md
02-vector-search/lessons/08-pgvector.md

In [7]:
sqlite_doc = next(
    doc for doc in documents
    if doc["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

print(sqlite_doc["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [8]:
doc_vector = embed.encode(sqlite_doc["content"])

In [9]:
similarity = query_vector.dot(doc_vector)

print(similarity)

0.36107027225589694


In [10]:
# Q3. Chunking and search by hand
from gitsource import chunk_documents

chunks = chunk_documents(
    documents,
    size=2000,
    step=1000
)

print(len(chunks))

295


In [11]:
chunks[0].keys()

dict_keys(['start', 'content', 'filename'])

In [12]:
chunks[0]

{'start': 0,
 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phon

In [13]:
texts = [chunk["content"] for chunk in chunks]

X = embed.encode_batch(texts)

print(X.shape)

(295, 384)


In [14]:
scores = X.dot(query_vector)

print(scores.shape)
print(scores.max())

(295,)
0.6489017718578813


In [15]:
best_idx = scores.argmax()

print(best_idx)
print(chunks[best_idx]["filename"])
print(chunks[best_idx]["start"])

94
02-vector-search/lessons/07-sqlitesearch-vector.md
1000


In [16]:
# Q4. Vector search with minsearch
from minsearch import VectorSearch

In [17]:
help(VectorSearch)

Help on class VectorSearch in module minsearch.vector:

class VectorSearch(builtins.object)
 |  VectorSearch(keyword_fields=None, numeric_fields=None, date_fields=None)
 |
 |  A vector search index using cosine similarity for vector search,
 |  exact matching for keyword fields, and range filters for numeric and date fields.
 |
 |  Takes a 2D numpy array of vectors and a list of payload documents, providing efficient
 |  similarity search with keyword, numeric, and date filtering capabilities.
 |
 |  Methods defined here:
 |
 |  __init__(self, keyword_fields=None, numeric_fields=None, date_fields=None)
 |      Initialize the VectorSearch index.
 |
 |      Args:
 |          keyword_fields (list, optional): List of keyword field names to index for exact matching. Defaults to empty list.
 |          numeric_fields (list, optional): List of numeric field names to index for range filters. Defaults to empty list.
 |          date_fields (list, optional): List of date field names to index for

In [19]:
import inspect
from minsearch import VectorSearch

print(inspect.signature(VectorSearch.fit))

(self, vectors, payload)


In [20]:
print(inspect.signature(VectorSearch.search))

(self, query_vector, filter_dict=None, num_results=10, output_ids=False)


In [21]:
from minsearch import VectorSearch

index = VectorSearch()

In [23]:
index.fit(X, chunks)

In [24]:
query = "What metric do we use to evaluate a search engine?"

query_vector = embed.encode(query)

In [25]:
results = index.search(query_vector)

In [26]:
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

In [27]:
# Q5. Text search vs vector search
from minsearch import Index

In [28]:
text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

In [29]:
text_index.fit(chunks)

In [30]:
query = "How do I store vectors in PostgreSQL?"

text_results = text_index.search(
    query=query,
    num_results=5
)

In [31]:
print("TEXT SEARCH")

for result in text_results:
    print(result["filename"])

TEXT SEARCH
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
